# **Smart Water Systems: Advanced Methods for Time-Series Forecasting**

## Learning goals
By the end of this notebook, students should be able to:

1. Build a **direct multi-step OLS forecast** with a separate model for each forecast horizon.
2. Explain the difference between **direct** and **recursive** multi-step forecasting.
3. Prepare horizon-specific predictors and targets using `shift()`.
4. Evaluate forecast accuracy using MAE and RMSE.

We will use **DMA 1 demand (L/s)** throughout the notebook.


In [ ]:
# imprort packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# **1. Load the DMA 1 data**

The CSV file contains timestamps, water demand, and air temperature. The temperature variable has already been added to the file, so we will read it directly rather than generate it in the notebook.


In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Define course folders
course_folder = '/content/drive/MyDrive/CE374U2_Fall2026'
data_folder = os.path.join(course_folder, 'data')
figures_folder = os.path.join(course_folder, 'figures')

file_name = '02_04_timeseries_forecast.csv'
data_file = os.path.join(data_folder, file_name)

data_file


In [ ]:
df = pd.read_csv(data_file)
df.head()


### **Rename and select the variables**

Long column names are useful in a data file, but shorter names make Python code easier to read.

`rename()` changes the column labels. It does **not** change the data.

`pd.to_datetime()` converts the timestamp from text into a datetime object. The argument `dayfirst=True` is important because these data use the European date format **day/month/year**.

Finally, `df[['timestamp', 'DMA1', 'Temperature']]` keeps only the columns needed in this notebook.

Notice the **double brackets**:

```python
df[['timestamp', 'DMA1']]
```

Double brackets return a DataFrame containing multiple selected columns. A single bracket such as `df['DMA1']` returns one Series.


In [ ]:
# rename columns

df = df.rename(columns={
    'Date-time CET-CEST (DD/MM/YYYY HH:mm)': 'timestamp',
    'DMA 1 (L/s)': 'DMA1',
    'Air temperature (C)': 'Temperature'
})

# convert to datetime

df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True)

df = df[['timestamp', 'DMA1', 'Temperature']]

df.head()


### **Check the data**

Before building a model, check the period of record and missing values. A regression model cannot be fit to rows in which the target or a required predictor is missing.


In [ ]:
print('Start:', df['timestamp'].min())
print('End:', df['timestamp'].max())
print('Number of records:', len(df))
print('Missing DMA1 values:', df['DMA1'].isna().sum())
print('Missing temperature values:', df['Temperature'].isna().sum())


# **2. Create training and testing periods**

In [ ]:
test_start = '2023-03-15'

train = df[df['timestamp'] < test_start].copy()
test = df[df['timestamp'] >= test_start].copy()

print('Training period:')
print(train['timestamp'].min(), 'to', train['timestamp'].max())

print('\nTest period:')
print(test['timestamp'].min(), 'to', test['timestamp'].max())

plt.figure(figsize=(12, 4))

plt.plot(train['timestamp'], train['DMA1'], label='Training')
plt.plot(test['timestamp'], test['DMA1'], label='Test')

plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('Training and test periods')
plt.legend()
plt.show()

# **3. Direct multi-step forecasting with OLS**

So far, each model has predicted **one time step at a time**. In the lag model, for example, the value of demand at time $t$ was predicted using observed lagged values such as $D_{t-1}$ and $D_{t-24}$.

Now suppose that at time $t$ we want to forecast the **next 24 hours at once**:

$$
D_{t+1},\;D_{t+2},\;\ldots,\;D_{t+24}
$$

There are different ways to make a multi-step forecast. Here we use **direct multi-step forecasting**.

### Direct versus recursive forecasting

A **recursive forecast** would first predict $D_{t+1}$, then use that prediction as an input for predicting $D_{t+2}$, and continue forward. Errors from early predictions can therefore propagate into later predictions.

A **direct forecast** does not feed predictions back into the model. Instead, we fit a different regression model for each forecast horizon:

- one model for 1 hour ahead,
- another model for 2 hours ahead,
- ...
- another model for 24 hours ahead.

For horizon $h$, we fit

$$
D_{t+h}
=
\beta_{0,h}
+\beta_{1,h}D_t
+\beta_{24,h}D_{t+h-24}
+\varepsilon_{t+h},
\qquad h=1,\ldots,24
$$

The coefficients have a subscript $h$ because **each forecast horizon has its own OLS model and its own coefficients**.

At forecast time $t$:

- $D_t$ is the most recent observed demand;
- $D_{t+h-24}$ is the demand at the same hour on the previous day relative to the target hour.

For horizons up to 24 hours, both predictors are already known at time $t$. We therefore do not need to use any predicted demand as an input.


# **Preparing the data**
### How do we create the horizon-specific variables?

For each horizon `h`, we reorganize the original time series into a regression table. The goal is to place the predictors available at the forecast origin and the future target on the **same row**.

The target is the future value $D_{t+h}$. In pandas, a negative shift moves future values upward so they align with the current forecast origin:

```python
df['DMA1'].shift(-h)
```

For example, when `h = 3`, the demand three hours in the future is placed on the current row.

The second predictor is $D_{t+h-24}$. This is the demand from the previous day at the **same hour as the target**. It is created with:

```python
df['DMA1'].shift(24 - h)
```

For example:

- `h = 1` uses `shift(23)`, corresponding to $D_{t-23}$;
- `h = 12` uses `shift(12)`, corresponding to $D_{t-12}$;
- `h = 24` uses `shift(0)`, so the previous-day target-hour value is $D_t$.

We also create `target_time` using `timestamp.shift(-h)`. This tells us when the value being predicted occurs and lets us separate training and testing data correctly.


## **Example: prepare the data for a 3-hour-ahead forecast**

In [ ]:
# Example: prepare the data for a 3-hour-ahead forecast

h = 3

horizon_df = pd.DataFrame({
    'timestamp': df['timestamp'],
    'Current_Demand': df[💡],
    'Target_Hour_Previous_Day': df[💡].shift(24 - 💡),
    'Target': df[💡].shift(💡),
    'target_time': df['timestamp'].shift(💡)
}).dropna().copy()

horizon_df.head(28)

## **Improt Linear Regression package**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

## **Example: Create one model for h = 3**

In [ ]:
# Forecast 3 hours ahead
h = 💡

# Each row represents a forecast made at time t.
# Current_Demand is D_t.
# Target_Hour_Previous_Day is D_(t+h-24).
# Target is D_(t+h).
horizon_df = pd.DataFrame({
    'timestamp': df['timestamp'],
    'Current_Demand': df['DMA1'],
    'Target_Hour_Previous_Day': df['DMA1'].shift(24 - h),
    'Target': df['DMA1'].shift(-h),
    'target_time': df['timestamp'].shift(-h)
}).dropna().copy()

# Training targets must occur before the testing period
train_h = horizon_df[horizon_df['target_time'] < test_start].copy()

# Testing targets occur during the testing period
test_h = horizon_df[horizon_df['target_time'] >= test_start].copy()

# x,y train
X_train = train_h[['💡','💡']]
y_train = train_h['💡']      # true values

# Fit one OLS model for this forecast horizon
model_h = LinearRegression()

model_h.fit(💡, 💡)


In [ ]:
# get beta_0

model_h.intercept_

In [ ]:
# get all other betas

model_h.coef_

## Predict one observation

In [ ]:
current_time = '2021-01-03 13:00'

In [ ]:
# get values for current time
current_row = horizon_df[horizon_df['timestamp'] == current_time].copy()

X = current_row[['Current_Demand','Target_Hour_Previous_Day']]



In [ ]:
# provide x

prediction = model_h.predict(X)

# predict y

prediction

In [ ]:
# verify using manual calculation

manual_prediction = model_h.intercept_+ model_h.coef_[0] * current_row['Current_Demand'].iloc[0] + model_h.coef_[1] * current_row['Target_Hour_Previous_Day'].iloc[0]
manual_prediction

In [ ]:
# actual value

current_row[['Target','target_time']]

# **Create different models**

The model above predicted a single point into the future at t+h. Next, we will fit 24 models for different h = 1, 2, ... 24.

### What does the `for` loop do?

A `for` loop repeats the same block of code for several values. Here,

```python
for h in range(1, max_horizon + 1):
```

sets `h` equal to 1, then 2, then 3, and continues through 24.

For every value of `h`, Python:

1. creates the correct future target,
2. creates the predictors that are available at the forecast origin,
3. fits a **new OLS model**,

The dictionary

```python
horizon_models[h] = model_h
```

stores each fitted model using its horizon as the key. For example, `horizon_models[6]` is the model trained specifically to forecast 6 hours ahead.

This is why this method is called **direct multi-step forecasting**. There are 24 separate regressions rather than one regression repeatedly feeding predictions back into itself.


In [ ]:

# Number of hours to forecast ahead
max_horizon = 24

# Store one fitted OLS model and one error value for each horizon
horizon_models = {}

for h in range(1, 💡 + 1):

    # Create new data for each model
    # Each row represents a forecast made at time t.
    # Current_Demand is D_t.
    # Target_Hour_Previous_Day is D_(t+h-24).
    # Target is D_(t+h).
    horizon_df = pd.DataFrame({
        'timestamp': df['timestamp'],
        'Current_Demand': df['DMA1'],
        'Target_Hour_Previous_Day': df['DMA1'].shift(24 - h),
        'Target': df['DMA1'].shift(-h),
        'target_time': df['timestamp'].shift(-h)
    }).dropna().copy()

    # Training targets must occur before the testing period
    train_h = horizon_df[horizon_df['target_time'] < test_start].copy()

    # x,y train
    X_train = train_h[['Current_Demand','Target_Hour_Previous_Day']]
    y_train = train_h['Target']      # true values

    # Fit one OLS model for this forecast horizon
    model_h = LinearRegression()

    model_h.fit(X_train, y_train)

    # Save the fitted model
    horizon_models[h] = model_h


In [ ]:
horizon_models[3].intercept_


In [ ]:
horizon_models[3].coef_

In [ ]:
# let's look at all the coefficients of all models

coefficient_results = []

for h in range(1, max_horizon + 1):
    model_h = horizon_models[h]

    coefficient_results.append({
        'Horizon': h,
        'Intercept': model_h.intercept_,
        'Current_Demand': model_h.coef_[0],
        'Target_Hour_Previous_Day': model_h.coef_[1]
    })

coefficient_results = pd.DataFrame(coefficient_results)

coefficient_results

# **Make one 24-hour forecast**

Next, we select **one current time, $t$**, and generate forecasts from 1 to 24 hours ahead from that same point.

We use the last observed hour immediately before the testing period as the current time. For each horizon $h$, we use the corresponding horizon-specific OLS model to predict demand at $t+h$. The predictors are the current demand at $t$ and the demand at the same target hour on the previous day, $t+h-24$:

$$
\hat{D}_{t+h} = f_h(D_t,\;D_{t+h-24})
$$

where $f_h$ is the OLS model trained specifically for forecast horizon $h$.

Notice that all 24 forecasts are made using information available at time $t$. The code never replaces an observed value with an earlier prediction, so the forecasts for each horizon are generated independently.

#**Example**

In [ ]:
# Find the current time t, the last observed hour before the testing period

# we'll use indices now instead of the dates so we can loop over multiple dates

# All 1- to 24-hour-ahead forecasts will be made from this same point in time
current_time = '2023-03-14 23:00'   # this is the last day of the training period

current_index = df.index[df['timestamp'] == current_time][0]

# Time and demand observed at the current time t
current_time = df.loc[current_index, 'timestamp']
current_demand = df.loc[current_index, 'DMA1']

print(current_index)
print(current_time)
print(current_demand)

In [ ]:
# Generate forecasts for hour 3 from current time
h = 3

# Row corresponding to the target time t+h
# For example, h = 3 means 3 hours after the current time
target_index = current_index + h

# Row corresponding to the same target hour on the previous day
# This is the demand at t+h-24
previous_day_index = target_index - 24

previous_day_demand = df.loc[previous_day_index, 'DMA1']

print(target_index)
print(previous_day_index)
print(previous_day_demand)

In [ ]:
df.head()

In [ ]:
# Create the predictor values for this forecast horizon
# Current_Demand is the demand at t and is the same for all horizons
# Target_Hour_Previous_Day is the demand at t+h-24 and changes with h
X_future = pd.DataFrame({
    'Current_Demand': [df.loc[current_index, 'DMA1']],
    'Target_Hour_Previous_Day': [df.loc[previous_day_index, 'DMA1']]
})

X_future.head()

In [ ]:
# Use the model trained specifically for horizon h
# For example, horizon_models[3] is the 3-hour-ahead model
predicted = horizon_models[h].predict(X_future)[0]

print(predicted)

In [ ]:
# Store the forecast time, observed demand, and predicted demand at t+h
forecast_rows = []
forecast_rows.append({
    'Horizon': h,
    'timestamp': df.loc[target_index, 'timestamp'],
    'Observed': df.loc[target_index, 'DMA1'],
    'Forecast': predicted
})

forecast_rows
# # Combine the 24 individual forecasts into one DataFrame
# forecast_24h = pd.DataFrame(forecast_rows)

# # Display the current time and the first five forecast horizons
# print('Current time:', current_time)
# forecast_24h.head()

# **24-hr forecast**
Next, we'll loop over all the models to get 24-hr forecast

In [ ]:
# Find the current time t, the last observed hour before the testing period
# All 1- to 24-hour-ahead forecasts will be made from this same point in time

current_time = '2023-03-14 23:00'   # this is the last day of the training period

current_index = df.index[df['timestamp'] == current_time][0]

# Time and demand observed at the current time t
current_time = df.loc[current_index, 'timestamp']

# Empty list to store the forecast for each horizon
forecast_rows = []

# Generate forecasts from 1 hour ahead to 24 hours ahead
for h in range(1, max_horizon + 1):

    # Row corresponding to the target time t+h
    # For example, h = 3 means 3 hours after the current time
    target_index = current_index + h

    # Row corresponding to the same target hour on the previous day
    # This is the demand at t+h-24
    previous_day_index = target_index - 24

    # Create the predictor values for this forecast horizon
    # Current_Demand is the demand at t and is the same for all horizons
    # Target_Hour_Previous_Day is the demand at t+h-24 and changes with h
    X_future = pd.DataFrame({
        'Current_Demand': [df.loc[current_index, 'DMA1']],
        'Target_Hour_Previous_Day': [df.loc[previous_day_index, 'DMA1']]
    })

    # Use the model trained specifically for horizon h
    # For example, horizon_models[3] is the 3-hour-ahead model
    predicted = horizon_models[h].predict(X_future)[0]

    # Store the forecast time, observed demand, and predicted demand at t+h
    forecast_rows.append({
        'Horizon': h,
        'timestamp': df.loc[target_index, 'timestamp'],
        'Observed': df.loc[target_index, 'DMA1'],
        'Forecast': predicted
    })

# Combine the 24 individual forecasts into one DataFrame
forecast_24h = pd.DataFrame(forecast_rows)

# Display the current time and the first five forecast horizons
print('Current time:', current_time)
forecast_24h.head()

### Plot the 24-hour direct forecast

This plot shows the 24 independently generated forecasts together as one forecast trajectory. The observed values are shown only for evaluation. They were **not** used to generate the predictions.


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(forecast_24h['timestamp'], forecast_24h['Observed'], marker='o', label='Observed')
plt.plot(forecast_24h['timestamp'], forecast_24h['Forecast'], marker='o', label='Direct forecast')
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('24-Hour Direct Multi-Step Forecast')
plt.legend()
plt.grid(True)
plt.show()

mae_24 = mean_absolute_error(forecast_24h['Observed'],forecast_24h['Forecast'])

rmse_24 = root_mean_squared_error(forecast_24h['Observed'],forecast_24h['Forecast'])


### **Important distinction**

The direct OLS approach builds a **separate** regression model for **every** forecast horizon. Each model uses predictors that are available at the forecast origin.



## **Try a different day**

## **How does forecast accuracy change with horizon?**
After training the models we can calculate RMSE for every forecast horizon. Plotting RMSE against horizon shows whether forecasts become less accurate as we look farther into the future.

In [ ]:
# Number of hours to forecast ahead
max_horizon = 24

# Store one fitted OLS model and one error value for each horizon
horizon_models = {}
horizon_results = []

for h in range(1, max_horizon + 1):

    # Create new data for each model
    # Each row represents a forecast made at time t.
    # Current_Demand is D_t.
    # Target_Hour_Previous_Day is D_(t+h-24).
    # Target is D_(t+h).
    horizon_df = pd.DataFrame({
        'timestamp': df['timestamp'],
        'Current_Demand': df['DMA1'],
        'Target_Hour_Previous_Day': df['DMA1'].shift(24 - h),
        'Target': df['DMA1'].shift(-h),
        'target_time': df['timestamp'].shift(-h)
    }).dropna().copy()

    # Training targets must occur before the testing period
    train_h = horizon_df[horizon_df['target_time'] < test_start].copy()

    # x,y train
    X_train = train_h[['Current_Demand','Target_Hour_Previous_Day']]
    y_train = train_h['Target']      # true values

    # Fit one OLS model for this forecast horizon
    model_h = LinearRegression()

    model_h.fit(X_train, y_train)

    # Save the fitted model
    horizon_models[h] = model_h

    # Generate predictions
    #----------------------------------------
    # Testing targets occur during the testing period
    test_h = horizon_df[horizon_df['target_time'] >= test_start].copy()

    # x,y test
    X_test = test_h[['Current_Demand','Target_Hour_Previous_Day']]
    y_test = test_h['Target']      # true values


    test_h['Predicted'] = model_h.predict(X_test)

    # Evaluate predictions
    mae_h = mean_absolute_error(test_h['Target'], test_h['Predicted'])

    rmse_h = root_mean_squared_error(test_h['Target'], test_h['Predicted'])

    # Save the performance metrics
    horizon_results.append({
        'Horizon': h,
        'MAE': mae_h,
        'RMSE': rmse_h,
        'Number_of_Test_Predictions': len(test_h)
    })

# Convert results to a DataFrame
horizon_results = pd.DataFrame(horizon_results)

horizon_results.head()

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(horizon_results['Horizon'], horizon_results['RMSE'], marker='o', label = 'RMSE')
#plt.plot(horizon_results['Horizon'], horizon_results['MAE'], marker='s', label = 'MAE')
plt.xlabel('Forecast horizon (hours ahead)')
plt.title('Direct Multi-Step Forecast Error by Horizon')
plt.xticks(range(1, max_horizon + 1, 2))
plt.legend()
plt.grid(True)
plt.show()

horizon_results


# **5. Key Python ideas from this notebook**

### `.copy()`
Creates a separate DataFrame that we can safely modify.

```python
new_df = old_df[condition].copy()
```

### `.shift(k)`
Moves a series by `k` rows and is useful for creating lagged predictors.

```python
df['lag_24'] = df['DMA1'].shift(24)
```

### `.dropna()`
Removes rows containing missing values in the selected DataFrame.

```python
model_df = df[['DMA1', 'lag_24']].dropna()
```

### `LinearRegression()`
Creates a linear regression model.

### `.fit(X, y)`
Estimates the model coefficients using predictor matrix `X` and target `y`.

### `.predict(X)`
Uses the fitted coefficients to calculate predicted values for new predictor data.

### Why is `X`  selected with double brackets?

Scikit-learn expects the predictors to be a two-dimensional table, even when there is only one predictor:

```python
X = df[['Temperature']]
```

The target is one-dimensional:

```python
y = df['DMA1']
```

Keeping this distinction in mind helps avoid many common scikit-learn errors.

### `.shift(-h)`
A negative shift aligns a future value with the current row. This is useful when creating a target for an $h$-step-ahead forecast.

```python
df['Target'] = df['DMA1'].shift(-h)
```

### `range()` and a `for` loop
`range(1, 25)` produces the integers 1 through 24. A `for` loop lets us repeat the model-building procedure for every forecast horizon.

```python
for h in range(1, 25):
    # create target and fit one model for horizon h
```

### Dictionaries
A dictionary stores objects using named or numbered keys. We used a dictionary to keep the fitted model for every horizon.

```python
horizon_models[h] = model_h
```

The 6-hour model can then be retrieved with `horizon_models[6]`.




# **6. Final discussion**

1. Why does direct multi-step OLS require a separate model for each forecast horizon?
2. What is the main difference between direct and recursive multi-step forecasting?
3. Why must we be careful not to use predictor values that occur after the forecast origin?